## import dependencies

Training script for Vision Transformer (ViT) model  
Data is Tiny Dataset of MNIST

In [ ]:
import torch
import torch.nn as nn
from vit_model import VisionTransformer
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# tensorboard is pytorch dashboard for training
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter()
# Hyperparameters
img_size = 32
patch_size = 8
in_channels = 1 # MNIST images are grayscale. Usually 3 channel for RGB images
embed_dim = 64
num_heads = 8
mlp_hidden_dim = 128 # embed_dim * 2 or * 4
num_layers = 2
num_classes = 10
dropout = 0.1

# Initialize the Vision Transformer model
model = VisionTransformer(
    img_size=img_size,
    patch_size=patch_size,
    in_channels=in_channels,
    embed_dim=embed_dim,
    num_heads=num_heads,
    mlp_hidden_dim=mlp_hidden_dim,
    num_layers=num_layers,
    num_classes=num_classes,
    dropout=dropout
)

# Print the model architecture
print(model)

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# train_loader for MNIST
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

# It will download MNIST data to ./data forlder when the first time running.
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Training loop (example)
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:  # Assuming you have a DataLoader named train_loader
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
    writer.add_scalar('Loss/train', loss.item(), epoch)

# Save the trained model
torch.save(model.state_dict(), "vit_model.pth")

# Evaluate the trained model on the test dataset
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")
writer.add_scalar('Accuracy/test', 100 * correct / total, num_epochs)
writer.close()

## Get started with TensorBoard

Run:  tensorboard --logdir=runs  
View: http://localhost:6006